In [4]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

FILE_NAME = "p0.8_mu0.2_l10_1"
emb_path = f"../pretrain/{FILE_NAME}.npy"
interaction_path = f"../syn_data/{FILE_NAME}.csv"
K = 5

X = np.load(emb_path).astype(np.float32)   # shape: [num_nodes, dim]


node_ids = np.arange(X.shape[0])

kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
labels = kmeans.fit_predict(X)

node2label = pd.DataFrame({
    "node": node_ids,
    "label": labels
})

inter_df = pd.read_csv(interaction_path)
inter_df = inter_df[["source", "destination", "timestamp"]].copy()

inter_df["source"] = inter_df["source"].astype(int)
inter_df["destination"] = inter_df["destination"].astype(int)

src_label_df = node2label.rename(columns={
    "node": "source",
    "label": "source_commu"
})
result_df = inter_df.merge(src_label_df, on="source", how="left")

dst_label_df = node2label.rename(columns={
    "node": "destination",
    "label": "destination_commu"
})
result_df = result_df.merge(dst_label_df, on="destination", how="left")

result_df = result_df[
    ["source", "destination", "timestamp", "source_commu", "destination_commu"]
].copy()

out_dir = os.path.join("../results", FILE_NAME)
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "ctdne.csv")
result_df.to_csv(out_path, index=False)

print(result_df.head(20))
print(f"saved to {out_path}")
print("rows =", len(result_df))
print("missing source_commu =", result_df["source_commu"].isna().sum())
print("missing destination_commu =", result_df["destination_commu"].isna().sum())

    source  destination  timestamp  source_commu  destination_commu
0        9           25          3             2                  1
1       14           41          7             3                  0
2       26           49          7             2                  0
3        8           43          7             2                  2
4       22           42          9             2                  2
5       23           29         14             0                  0
6       17           41         16             2                  0
7        6           46         20             2                  3
8       25           32         22             1                  4
9       17           19         24             2                  1
10      20           32         25             4                  4
11      12           27         25             2                  4
12      28           44         25             2                  2
13      16           30         26             0

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

SYN_DATA_DIR = "syn_data/"
PRETRAIN_DIR = "pretrain/"
RESULTS_DIR = "results/"
K = 5

csv_files = sorted(glob.glob(os.path.join(SYN_DATA_DIR, "*.csv")))
print(f"Found {len(csv_files)} csv files.")

for interaction_path in csv_files:
    FILE_NAME = os.path.splitext(os.path.basename(interaction_path))[0]
    emb_path = os.path.join(PRETRAIN_DIR, f"{FILE_NAME}.npy")

    print(f"\nProcessing: {FILE_NAME}")

    if not os.path.exists(emb_path):
        print(f"  Skip: embedding file not found -> {emb_path}")
        continue

    X = np.load(emb_path).astype(np.float32)   # shape: [num_nodes, dim]
    node_ids = np.arange(X.shape[0])

    kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
    labels = kmeans.fit_predict(X)

    node2label = pd.DataFrame({
        "node": node_ids,
        "label": labels
    })

    inter_df = pd.read_csv(interaction_path)
    inter_df = inter_df[["source", "destination", "timestamp"]].copy()
    inter_df["source"] = inter_df["source"].astype(int)
    inter_df["destination"] = inter_df["destination"].astype(int)

    src_label_df = node2label.rename(columns={
        "node": "source",
        "label": "source_commu"
    })
    result_df = inter_df.merge(src_label_df, on="source", how="left")

    dst_label_df = node2label.rename(columns={
        "node": "destination",
        "label": "destination_commu"
    })
    result_df = result_df.merge(dst_label_df, on="destination", how="left")

    result_df = result_df[
        ["source", "destination", "timestamp", "source_commu", "destination_commu"]
    ].copy()

    out_dir = os.path.join(RESULTS_DIR, FILE_NAME)
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, "ctdne.csv")
    result_df.to_csv(out_path, index=False)

    print(result_df.head(5))
    print(f"  saved to {out_path}")
    print(f"  rows = {len(result_df)}")
    print(f"  missing source_commu = {result_df['source_commu'].isna().sum()}")
    print(f"  missing destination_commu = {result_df['destination_commu'].isna().sum()}")

print("\nDone.")

Found 500 csv files.

Processing: p0.85_mu0.05_l10_1
   source  destination  timestamp  source_commu  destination_commu
0      34           45          0             2                  0
1      13           34          5             0                  2
2      14           34          6             0                  2
3      12           33          7             0                  0
4       4           47          7             1                  1
  saved to results/p0.85_mu0.05_l10_1/ctdne.csv
  rows = 2485
  missing source_commu = 0
  missing destination_commu = 0

Processing: p0.85_mu0.05_l10_2
   source  destination  timestamp  source_commu  destination_commu
0      36           38          0             2                  1
1      13           47          5             1                  4
2      15           24          6             2                  2
3      12           42          7             1                  4
4       5           39          7             2          